# Prueba de Hipótesis 1: Comparación de diámetros entre especies de árboles
## Prueba de Kruskal-Wallis

In [ ]:
# Importación de bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Carga de datos
df = pd.read_csv('../data/datos_arbolado_clean.csv')

# Visualización de las primeras filas e información básica
print("Dimensiones del dataset:", df.shape)
print("\nPrimeras filas:")
df.head()

In [ ]:
# Información sobre las variables de interés
print("Información sobre las variables clave:")
print(f"Diámetro promedio: {df['diametro_altura_pecho'].mean():.2f} cm")
print(f"Diámetro mediano: {df['diametro_altura_pecho'].median():.2f} cm")
print(f"Número de especies únicas: {df['nombre_cientifico'].nunique()}")

print("\nEspecies más frecuentes:")
especies_counts = df['nombre_cientifico'].value_counts()
print(especies_counts.head(10))

In [ ]:
# Selección de especies con al menos 10 observaciones para una mejor fiabilidad estadística
especies_frecuentes = especies_counts[especies_counts >= 10].index
df_filtrado = df[df['nombre_cientifico'].isin(especies_frecuentes)]

print(f"Número de especies retenidas: {len(especies_frecuentes)}")
print(f"Número total de observaciones: {len(df_filtrado)}")
print("\nEspecies retenidas:")
print(especies_frecuentes.tolist())

In [ ]:
# Estadísticas descriptivas por especie
print("ESTADÍSTICAS DESCRIPTIVAS POR ESPECIE")
print("=" * 50)

stats_especies = df_filtrado.groupby('nombre_cientifico')['diametro_altura_pecho'].agg([
    'count', 'mean', 'median', 'std', 'min', 'max'
]).round(2)

stats_especies_sorted = stats_especies.sort_values('mean', ascending=False)
stats_especies_sorted

In [ ]:
# Visualización 1: Boxplot de los diámetros por especie
plt.figure(figsize=(14, 8))

# Ordenar por diámetro mediano
especies_ordenadas = df_filtrado.groupby('nombre_cientifico')['diametro_altura_pecho'].median().sort_values(ascending=False).index

sns.boxplot(data=df_filtrado, 
            x='diametro_altura_pecho', 
            y='nombre_cientifico',
            order=especies_ordenadas)

plt.title('Distribución de Diámetros por Especie de Árbol', fontsize=16, fontweight='bold')
plt.xlabel('Diámetro a la Altura del Pecho (cm)', fontsize=12)
plt.ylabel('Especie', fontsize=12)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualización 2: Violin plot para ver la densidad de distribución
plt.figure(figsize=(14, 8))

sns.violinplot(data=df_filtrado, 
               x='diametro_altura_pecho', 
               y='nombre_cientifico',
               order=especies_ordenadas)

plt.title('Distribución de Diámetros por Especie (Violin Plot)', fontsize=16, fontweight='bold')
plt.xlabel('Diámetro a la Altura del Pecho (cm)', fontsize=12)
plt.ylabel('Especie', fontsize=12)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Preparación de datos para la prueba de Kruskal-Wallis
grupos = []
for especie in especies_frecuentes:
    grupo = df_filtrado[df_filtrado['nombre_cientifico'] == especie]['diametro_altura_pecho']
    grupos.append(grupo)

print(f"Número de grupos (especies) para la prueba: {len(grupos)}")
print(f"Tamaños de los grupos: {[len(g) for g in grupos]}")

In [ ]:
# Prueba de Kruskal-Wallis
print("PRUEBA DE KRUSKAL-WALLIS")
print("=" * 40)

statistic, p_value = stats.kruskal(*grupos)

print(f"Estadística H de Kruskal-Wallis: {statistic:.4f}")
print(f"Valor p: {p_value:.10f}")
print(f"Valor p (notación científica): {p_value:.2e}")

# Interpretación
alpha = 0.05
print("\nINTERPRETACIÓN:")
print("-" * 20)
if p_value < alpha:
    print(f"✓ Se rechaza la hipótesis nula (p-value < {alpha})")
    print("✓ Existen diferencias estadísticamente significativas entre los diámetros de las diferentes especies de árboles")
else:
    print(f"✗ No se puede rechazar la hipótesis nula (p-value ≥ {alpha})")
    print("✗ No hay diferencias estadísticamente significativas entre los diámetros de las diferentes especies")

In [ ]:
# Prueba de normalidad (Shapiro-Wilk) sobre los residuos para verificar los supuestos
print("\nPRUEBA DE NORMALIDAD (SHAPIRO-WILK) SOBRE LOS DIÁMETROS")
print("=" * 55)

# Prueba sobre el conjunto de datos
stat_shapiro, p_shapiro = stats.shapiro(df_filtrado['diametro_altura_pecho'])
print(f"Prueba de Shapiro-Wilk sobre todos los diámetros:")
print(f"  Estadística: {stat_shapiro:.4f}")
print(f"  Valor p: {p_shapiro:.10f}")

if p_shapiro < 0.05:
    print("  ✓ Los datos no siguen una distribución normal")
    print("  ✓ La prueba de Kruskal-Wallis es apropiada")
else:
    print("  ✗ Los datos podrían seguir una distribución normal")

In [ ]:
# Pruebas post-hoc de Dunn si la prueba de Kruskal-Wallis es significativa
if p_value < alpha:
    print("\nPRUEBAS POST-HOC DE DUNN (comparaciones múltiples)")
    print("=" * 45)
    
    # Instalación e importación de scikit-posthocs si es necesario
    try:
        import scikit_posthocs as sp
        
        # Preparación de datos para Dunn
        dunn_data = []
        dunn_labels = []
        
        for i, especie in enumerate(especies_frecuentes):
            diametros = df_filtrado[df_filtrado['nombre_cientifico'] == especie]['diametro_altura_pecho']
            dunn_data.extend(diametros)
            dunn_labels.extend([especie] * len(diametros))
        
        # Prueba de Dunn
        dunn_result = sp.posthoc_dunn(
            pd.DataFrame({'diametro': dunn_data, 'especie': dunn_labels}), 
            val_col='diametro', 
            group_col='especie', 
            p_adjust='bonferroni'
        )
        
        print("Matriz de p-values ajustadas (prueba de Dunn):")
        print(dunn_result.round(4))
        
    except ImportError:
        print("scikit-posthocs no está instalado. Instalar con: pip install scikit-posthocs")
else:
    print("\nLa prueba post-hoc no es necesaria porque la prueba de Kruskal-Wallis no es significativa")

In [ ]:
# Visualización de las diferencias con un gráfico de barras de los diámetros promedios
plt.figure(figsize=(12, 6))

diametros_medios = df_filtrado.groupby('nombre_cientifico')['diametro_altura_pecho'].mean().sort_values(ascending=False)

bars = plt.bar(range(len(diametros_medios)), diametros_medios.values)
plt.xticks(range(len(diametros_medios)), diametros_medios.index, rotation=45, ha='right')

# Adición de valores en las barras
for i, bar in enumerate(bars):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{height:.1f}', ha='center', va='bottom', fontweight='bold')

plt.title('Diámetro Promedio por Especie de Árbol', fontsize=16, fontweight='bold')
plt.ylabel('Diámetro Promedio (cm)', fontsize=12)
plt.xlabel('Especie', fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Resumen de resultados
print("RESUMEN DE RESULTADOS")
print("=" * 30)
print(f"• Hipótesis probada: Los diámetros difieren según las especies de árboles")
print(f"• Prueba utilizada: Kruskal-Wallis (no paramétrica)")
print(f"• Número de especies comparadas: {len(especies_frecuentes)}")
print(f"• Resultado de la prueba: {'SIGNIFICATIVO' if p_value < alpha else 'NO SIGNIFICATIVO'}")
print(f"• P-value: {p_value:.2e}")
print(f"• Especie con mayor diámetro promedio: {diametros_medios.index[0]} ({diametros_medios.iloc[0]:.1f} cm)")
print(f"• Especie con menor diámetro promedio: {diametros_medios.index[-1]} ({diametros_medios.iloc[-1]:.1f} cm)")

if p_value < alpha:
    print("\nCONCLUSIÓN: Existen diferencias significativas en los diámetros según las especies.")
    print("Esto podría explicarse por diferencias en:")
    print("  - La velocidad de crecimiento de las especies")
    print("  - La edad de los árboles plantados")
    print("  - Las condiciones de crecimiento")
else:
    print("\nCONCLUSIÓN: No se detectó ninguna diferencia significativa entre las especies.")